# 2026 호르무즈 해협 영어 해외언론 텍스트마이닝 수집

**기간:** 2026-01-01 ~ 2026-08-31  
**핵심 키워드:** `"Strait of Hormuz"`, `"Hormuz Strait"`  
**수집 구조:** NewsCatcher → URL 중복 제거 → **메타데이터 품질 필터** → Diffbot Article → 본문 품질 필터 → 텍스트마이닝

이 노트북은 대규모 NewsCatcher 후보군을 먼저 정리한 뒤,
필터를 통과한 기사에만 Diffbot Article API를 호출하도록 구성되어 있습니다.

> API 키는 코드에 직접 입력하지 않고 `.env` 파일의 환경변수로 넣습니다.
> 본문 수집 전에는 본문 길이를 정확히 알 수 없으므로,
> 본문 길이/실제 언어 검증은 Diffbot 수집 후 별도로 수행합니다.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

print("현재 작업 폴더:", PROJECT_ROOT)
print("프로젝트 폴더 존재:", PROJECT_ROOT.name == "Hormuz_Research")
print("데이터 폴더 존재:", (PROJECT_ROOT / "hormuz_2026_data").exists())

현재 작업 폴더: c:\Users\황태하\Hormuz_Research
프로젝트 폴더 존재: True
데이터 폴더 존재: True


## 0. 설치

Jupyter/VS Code에서 최초 1회만 실행합니다.


In [1]:
%pip install -q requests pandas openpyxl tqdm python-dotenv nltk scikit-learn matplotlib wordcloud langdetect


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 라이브러리와 기본 설정

NewsCatcher API는 `https://v3-api.newscatcherapi.com/api/search`를 사용하고,
API 키는 `x-api-token` 헤더로 전달합니다.

Diffbot Article API는 원문 URL에서 제목, 본문, 날짜, 저자, 언론사명,
언론사 국가/지역, 언어 등을 추출할 수 있으며 댓글도 `discussion`으로 반환할 수 있습니다.


In [2]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import requests
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

NEWSCATCHER_API_KEY = os.getenv("NEWSCATCHER_API_KEY")
DIFFBOT_API_KEY = os.getenv("DIFFBOT_API_KEY")

if not NEWSCATCHER_API_KEY:
    raise ValueError(
        "NEWSCATCHER_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

if not DIFFBOT_API_KEY:
    raise ValueError(
        "DIFFBOT_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

# API 키 자체는 출력하지 않습니다.
print("API 키 로드 완료")
print(f"NewsCatcher API: {'설정됨' if NEWSCATCHER_API_KEY else '없음'}")
print(f"Diffbot API: {'설정됨' if DIFFBOT_API_KEY else '없음'}")

START_DATE = "2026-01-01"
END_DATE = "2026-08-31"

QUERIES = [
    '"Strait of Hormuz"',
    '"Hormuz Strait"',
]

NEWS_URL = "https://v3-api.newscatcherapi.com/api/search"
DIFFBOT_ARTICLE_URL = "https://api.diffbot.com/v3/article"
DIFFBOT_DISCUSSION_URL = "https://api.diffbot.com/v3/discussion"

DATA_DIR = Path("hormuz_2026_data")
DATA_DIR.mkdir(exist_ok=True)

RAW_NEWS_FILE = DATA_DIR / "01_newscatcher_raw.csv"
ARTICLES_FILE = DATA_DIR / "02_articles.csv"
COMMENTS_FILE = DATA_DIR / "03_comments.csv"
FAILED_FILE = DATA_DIR / "04_failed_urls.csv"


API 키 로드 완료
NewsCatcher API: 설정됨
Diffbot API: 설정됨


c:\Users\황태하\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. API 연결 테스트

실제 대량 수집 전에 **각 API 키가 정상인지 먼저 확인**합니다.


In [3]:
def test_newscatcher():
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }
    payload = {
        "q": '"Strait of Hormuz"',
        "lang": "en",
        "from_": START_DATE,
        "to_": END_DATE,
        "page_size": 1,
    }
    r = requests.post(NEWS_URL, headers=headers, json=payload, timeout=60)
    print("NewsCatcher:", r.status_code)
    r.raise_for_status()
    data = r.json()
    print("total_hits:", data.get("total_hits"))
    return data

def test_diffbot(test_url="https://www.bbc.com/news"):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": test_url,
        "discussion": "false",
    }
    r = requests.get(DIFFBOT_ARTICLE_URL, params=params, timeout=60)
    print("Diffbot:", r.status_code)
    r.raise_for_status()
    return r.json()

nc_test = test_newscatcher()


NewsCatcher: 200
total_hits: 10000


## 3. URL 정규화

검색어가 여러 개이면 같은 기사가 반복될 수 있으므로 URL을 정규화한 뒤 중복 제거합니다.


In [4]:
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "gclid", "fbclid", "mc_cid", "mc_eid"
}

def normalize_url(url):
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()
    parts = urlsplit(url)

    if parts.scheme not in {"http", "https"}:
        return None

    query_items = []
    for item in parts.query.split("&") if parts.query else []:
        if "=" in item:
            k, v = item.split("=", 1)
        else:
            k, v = item, ""
        if k.lower() not in TRACKING_PARAMS:
            query_items.append(f"{k}={v}")

    clean = urlunsplit((
        parts.scheme.lower(),
        parts.netloc.lower(),
        parts.path.rstrip("/") or "/",
        "&".join(query_items),
        ""
    ))
    return clean


## 4. NewsCatcher 기사 목록 수집

- 영어: `lang=en`
- 연구 기간: 2026-01-01 ~ 2026-08-31
- 페이지 크기: 최대 1000
- 검색어별 결과를 합친 뒤 URL 기준으로 중복 제거
- 검색 결과가 많으면 **날짜 구간을 자동으로 분할**합니다.

### 왜 날짜 분할이 필요한가?

NewsCatcher는 한 검색 조건에서 반환 가능한 기사 수에 상한이 있습니다.
따라서 2026-01-01 ~ 2026-08-31을 한 번에 검색하면 최신 기사부터 채워지고
과거 기사가 잘릴 수 있습니다.

이 코드는 먼저 전체 구간의 `total_hits`를 확인하고,
안전 기준을 초과하면 월/주/일 단위로 자동 분할하여 다시 검색합니다.


In [5]:
def _request_newscatcher(payload, max_retries=5):
    """NewsCatcher 요청 + 일시적 오류(429/5xx) 재시도."""
    headers = {
        "x-api-token": NEWSCATCHER_API_KEY,
        "Content-Type": "application/json",
    }

    for attempt in range(max_retries):
        try:
            r = requests.post(
                NEWS_URL,
                headers=headers,
                json=payload,
                timeout=90,
            )

            if r.status_code == 429:
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        "NewsCatcher rate limit(429)입니다. "
                        "요금제 제한 또는 요청량을 확인하세요."
                    )
                wait = min(60, 2 ** attempt * 2)
                print(f"  429 발생 → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            if 500 <= r.status_code < 600:
                if attempt == max_retries - 1:
                    r.raise_for_status()
                wait = min(60, 2 ** attempt * 2)
                print(f"  서버 오류({r.status_code}) → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except requests.RequestException:
            if attempt == max_retries - 1:
                raise
            wait = min(60, 2 ** attempt * 2)
            time.sleep(wait)

    raise RuntimeError("NewsCatcher 요청에 실패했습니다.")


def newscatcher_search(
    q,
    from_date=START_DATE,
    to_date=END_DATE,
    page=1,
    page_size=1000,
):
    payload = {
        "q": q,
        "lang": "en",
        "from_": from_date,
        "to_": to_date,
        "page": page,
        "page_size": page_size,
        "exclude_duplicates": True,
        "sort_by": "date",
    }
    return _request_newscatcher(payload)


def _date_range_days(from_date, to_date):
    start = pd.Timestamp(from_date)
    end = pd.Timestamp(to_date)
    return (end - start).days + 1


# 한 검색 구간에 너무 많은 기사가 있으면 더 작은 날짜 구간으로 분할합니다.
# 10,000건 제한에 정확히 걸리지 않도록 9,000건을 안전 기준으로 사용합니다.
NEWS_MAX_SAFE_HITS = 9000


def collect_newscatcher_chunk(
    q,
    from_date,
    to_date,
    depth=0,
):
    """한 날짜 구간을 검색하고, 결과가 많으면 재귀적으로 날짜를 쪼갭니다."""
    indent = "  " * depth

    first = newscatcher_search(
        q,
        from_date=from_date,
        to_date=to_date,
        page=1,
        page_size=1000,
    )

    total_hits = int(first.get("total_hits") or 0)
    total_pages = int(first.get("total_pages") or 1)

    print(
        f"{indent}[{q}] {from_date} ~ {to_date} | "
        f"hits={total_hits:,} | pages={total_pages}"
    )

    # 검색 결과가 안전 기준 이내이면 정상적으로 페이지를 모두 가져옵니다.
    if total_hits <= NEWS_MAX_SAFE_HITS:
        rows = first.get("articles", [])

        for page in range(2, total_pages + 1):
            data = newscatcher_search(
                q,
                from_date=from_date,
                to_date=to_date,
                page=page,
                page_size=1000,
            )
            rows.extend(data.get("articles", []))
            time.sleep(0.3)

        return rows

    # 하루까지 쪼갰는데도 너무 많다면 해당 API 검색 조건에서는
    # 더 이상 날짜로 분할할 수 없으므로 오류를 알립니다.
    if _date_range_days(from_date, to_date) <= 1:
        raise RuntimeError(
            f"하루 단위에서도 {total_hits:,}건이 검색됩니다: "
            f"{q} / {from_date} ~ {to_date}"
        )

    start = pd.Timestamp(from_date)
    end = pd.Timestamp(to_date)
    mid = start + (end - start) // 2

    left_from = start.strftime("%Y-%m-%d")
    left_to = mid.strftime("%Y-%m-%d")
    right_from = (mid + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    right_to = end.strftime("%Y-%m-%d")

    print(
        f"{indent}→ 결과가 많아 날짜를 분할합니다: "
        f"{left_from}~{left_to} / {right_from}~{right_to}"
    )

    left_rows = collect_newscatcher_chunk(
        q, left_from, left_to, depth + 1
    )
    right_rows = collect_newscatcher_chunk(
        q, right_from, right_to, depth + 1
    )

    return left_rows + right_rows


def collect_newscatcher(q, from_date=START_DATE, to_date=END_DATE):
    return collect_newscatcher_chunk(q, from_date, to_date)


# 검색어별로 전체 기간을 수집합니다.
all_raw = []

for q in QUERIES:
    try:
        rows = collect_newscatcher(q)
        print(f"[{q}] 최종 수집: {len(rows):,}건")
        all_raw.extend(rows)
    except Exception as e:
        print(f"검색 실패: {q} -> {e}")

raw_df = pd.json_normalize(all_raw)

print("검색어 통합 raw 기사 수:", len(raw_df))

# API 원본은 그대로 저장해 두고, 이후 단계에서 URL 중복을 제거합니다.
raw_df.to_csv(RAW_NEWS_FILE, index=False, encoding="utf-8-sig")

raw_df.head()


["Strait of Hormuz"] 2026-01-01 ~ 2026-08-31 | hits=10,000 | pages=10
→ 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-05-02 / 2026-05-03~2026-08-31
  ["Strait of Hormuz"] 2026-01-01 ~ 2026-05-02 | hits=10,000 | pages=10
  → 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-03-02 / 2026-03-03~2026-05-02
    ["Strait of Hormuz"] 2026-01-01 ~ 2026-03-02 | hits=10,000 | pages=10
    → 결과가 많아 날짜를 분할합니다: 2026-01-01~2026-01-31 / 2026-02-01~2026-03-02
      ["Strait of Hormuz"] 2026-01-01 ~ 2026-01-31 | hits=2,714 | pages=3
      ["Strait of Hormuz"] 2026-02-01 ~ 2026-03-02 | hits=10,000 | pages=10
      → 결과가 많아 날짜를 분할합니다: 2026-02-01~2026-02-15 / 2026-02-16~2026-03-02
        ["Strait of Hormuz"] 2026-02-01 ~ 2026-02-15 | hits=4,045 | pages=5
        ["Strait of Hormuz"] 2026-02-16 ~ 2026-03-02 | hits=10,000 | pages=10
        → 결과가 많아 날짜를 분할합니다: 2026-02-16~2026-02-23 / 2026-02-24~2026-03-02
          ["Strait of Hormuz"] 2026-02-16 ~ 2026-02-23 | hits=4,968 | pages=5
          ["Strait of Hormuz"] 2026-02-24 ~ 2026-03-

,title,author,authors,journalists,published_date,published_date_precision,updated_date,updated_date_precision,link,domain_url,...,word_count,is_opinion,twitter_account,all_links,all_domain_links,id,score,duplicate_count,duplicate_articles_group_id,is_content_truncated
0,US warns Iran against unsafe actions during na...,Khaleejtimes,"[Khaleejtimes, system]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/US_warn...,wn.com,...,65,False,@worldnewsdotcom,"[https://travelagents.com/, https://www.clevel...","[travelagents.com, globalweather.com, emission...",0b08bbab067093c463795c247d0aa5d0,13.954778,0.0,bc5fb181e0a94626bb3782ff5a21045f,True
1,"As Iran–US tensions soar, Strait of Hormuz cau...",The Siasat Daily,"[The Siasat Daily, system]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/As_Iran...,wn.com,...,79,False,@worldnewsdotcom,[https://www.publicnow.com/view/478346E02E8C8A...,"[dailymail.co.uk, dubai.com, cities.com, wages...",4bb0a43d1de1f82a121c877b8034a9eb,13.917098,0.0,220b3a2d4f534a10af6b2bf3083a33d6,True
2,What to know about the Strait of Hormuz as Ira...,JON GAMBRELL,[JON GAMBRELL],[Jon Gambrell],2026-01-31 00:00:00,date,2026-01-31 00:00:00,date,https://sitkasentinel.com/stories/what-to-know...,sitkasentinel.com,...,35,False,NaN,[],[],fb5e16653f0da485e647950f55a5141e,13.349553,0.0,a15836cc7c3348a092ff08a9ced55698,True
3,Iran's Foreign Minister Criticizes US Military...,,[],[],2026-01-31 00:00:00,date,NaN,NaN,https://turkiyenewupdates.com/irans-foreign-mi...,turkiyenewupdates.com,...,260,False,NaN,[],[],4a15cf9d2ae915b7ee21ba86c0c33803,13.323380,0.0,c41eedf2a44b48a4a32cb2d64198d724,True
4,Strait of Hormuz becomes centre of Iran's mili...,Hindustan Times,"[system, Hindustan Times]",[],2026-01-31 00:00:00,date,NaN,NaN,https://article.wn.com/view/2026/01/31/Strait_...,wn.com,...,8,False,@worldnewsdotcom,[https://www.independent.co.uk/news/world/midd...,"[students.com, ndtv.com, worldphotos.com, duba...",e2abf87c8b2ec60d9ce90c0d75414b99,12.836854,0.0,5a8a3fe4b7944189a9b51a91324080f5,True


## 5. NewsCatcher 결과를 연구용 기사 목록으로 정리

API 응답 구조가 계정/버전에 따라 일부 달라질 수 있으므로,
아래 코드는 자주 사용되는 필드 후보를 순서대로 찾아 사용합니다.


In [11]:
# ============================================================
# 5. 기존 NewsCatcher CSV → 연구용 article_index
# ============================================================
# 이 셀은 독립적으로 실행할 수 있습니다.
#
# 사용 데이터:
# C:\Users\황태하\Hormuz_Research\hormuz_2026_data\
# 01_newscatcher_raw.csv
#
# 중요:
# - NewsCatcher API 재호출 없음
# - 기존 CSV만 읽음
# - normalize_url 포함
# - make_article_index 포함
# ============================================================

from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
import hashlib
import pandas as pd


# ------------------------------------------------------------
# 1. 실제 데이터 위치
# ------------------------------------------------------------

DATA_DIR = Path(
    r"C:\Users\황태하\Hormuz_Research\hormuz_2026_data"
)

RAW_NEWS_FILE = DATA_DIR / "01_newscatcher_raw.csv"


print("=" * 70)
print("기존 NewsCatcher 데이터 확인")
print("=" * 70)

print("데이터 폴더:")
print(DATA_DIR)

print()
print("원본 CSV:")
print(RAW_NEWS_FILE)


# ------------------------------------------------------------
# 2. 폴더 확인
# ------------------------------------------------------------

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"\n데이터 폴더가 존재하지 않습니다.\n\n"
        f"{DATA_DIR}"
    )


# ------------------------------------------------------------
# 3. 원본 CSV 확인
# ------------------------------------------------------------

if not RAW_NEWS_FILE.exists():

    files = list(DATA_DIR.iterdir())

    print("\n현재 데이터 폴더에 존재하는 파일:")

    for file in files:
        print(" -", file.name)

    raise FileNotFoundError(
        f"\n01_newscatcher_raw.csv를 찾을 수 없습니다.\n\n"
        f"확인한 경로:\n"
        f"{RAW_NEWS_FILE}"
    )


# ------------------------------------------------------------
# 4. URL 정규화 함수
# ------------------------------------------------------------

def normalize_url(url):
    """
    기사 URL을 비교하기 위한 기본 정규화.

    제거:
    - 앞뒤 공백
    - fragment (#...)
    - URL 마지막 slash

    유지:
    - http / https
    - query string

    주의:
    URL 자체를 임의로 변경하지 않기 위해
    query parameter는 제거하지 않습니다.
    """

    if pd.isna(url):
        return None

    url = str(url).strip()

    if not url:
        return None

    try:
        parsed = urlsplit(url)

        if parsed.scheme not in {"http", "https"}:
            return None

        if not parsed.netloc:
            return None

        # fragment 제거
        normalized = urlunsplit((
            parsed.scheme.lower(),
            parsed.netloc.lower(),
            parsed.path.rstrip("/"),
            parsed.query,
            ""
        ))

        return normalized

    except Exception:
        return None


# ------------------------------------------------------------
# 5. 기존 NewsCatcher CSV 읽기
# ------------------------------------------------------------

raw_df = pd.read_csv(
    RAW_NEWS_FILE,
    encoding="utf-8-sig",
    low_memory=False
)


print()
print("=" * 70)
print("CSV 로드 완료")
print("=" * 70)

print(f"원본 레코드 수: {len(raw_df):,}")
print(f"컬럼 수: {len(raw_df.columns)}")

print()
print("컬럼 목록:")
print(raw_df.columns.tolist())


# ------------------------------------------------------------
# 6. 컬럼 존재 여부 확인
# ------------------------------------------------------------

if "url" not in raw_df.columns and "link" not in raw_df.columns:

    raise KeyError(
        "\n원본 CSV에서 기사 URL 컬럼을 찾을 수 없습니다.\n"
        "확인된 컬럼:\n"
        + "\n".join(
            f" - {c}"
            for c in raw_df.columns
        )
    )


# ------------------------------------------------------------
# 7. 보조 함수
# ------------------------------------------------------------

def first_existing(row, candidates, default=None):

    for c in candidates:

        if (
            c in row.index
            and pd.notna(row[c])
            and str(row[c]).strip()
        ):
            return row[c]

    return default


# ------------------------------------------------------------
# 8. 연구용 article_index 생성
# ------------------------------------------------------------

def make_article_index(df):

    records = []

    for _, row in df.iterrows():

        # --------------------------------------------
        # URL 찾기
        # --------------------------------------------

        url = first_existing(
            row,
            [
                "link",
                "url",
                "clean_url",
                "canonical_url",
                "parent_url"
            ]
        )

        if not url:
            continue

        # --------------------------------------------
        # URL 정규화
        # --------------------------------------------

        url = normalize_url(url)

        if not url:
            continue

        # --------------------------------------------
        # 기사 메타데이터 추출
        # --------------------------------------------

        records.append({

            "title_search": first_existing(
                row,
                ["title"]
            ),

            "description_search": first_existing(
                row,
                [
                    "summary",
                    "excerpt",
                    "content",
                    "description"
                ]
            ),

            "source_search": first_existing(
                row,
                [
                    "clean_url",
                    "domain_url",
                    "source_url",
                    "source_name",
                    "source"
                ]
            ),

            "published_search": first_existing(
                row,
                [
                    "published_date",
                    "published_at",
                    "pub_date"
                ]
            ),

            "url": url,

            "language_search": first_existing(
                row,
                [
                    "language",
                    "lang"
                ],
                "en"
            )
        })


    # ----------------------------------------------------
    # DataFrame 생성
    # ----------------------------------------------------

    out = pd.DataFrame(records)


    if out.empty:
        raise ValueError(
            "article_index가 비어 있습니다.\n"
            "원본 CSV의 URL 데이터를 확인해야 합니다."
        )


    # ----------------------------------------------------
    # URL 기준 중복 제거
    # ----------------------------------------------------

    before_dedup = len(out)

    out = (
        out
        .drop_duplicates(subset=["url"])
        .reset_index(drop=True)
    )

    after_dedup = len(out)


    # ----------------------------------------------------
    # article_id 생성
    # ----------------------------------------------------

    out["article_id"] = [
        hashlib.sha1(
            url.encode("utf-8")
        ).hexdigest()[:16]
        for url in out["url"]
    ]


    return out, before_dedup, after_dedup


# ------------------------------------------------------------
# 9. article_index 생성
# ------------------------------------------------------------

article_index, before_dedup, after_dedup = (
    make_article_index(raw_df)
)


# ------------------------------------------------------------
# 10. 결과 출력
# ------------------------------------------------------------

print()
print("=" * 70)
print("article_index 생성 완료")
print("=" * 70)

print(
    f"원본 NewsCatcher 레코드: "
    f"{len(raw_df):,}"
)

print(
    f"URL 정규화 후 레코드: "
    f"{before_dedup:,}"
)

print(
    f"URL 중복 제거 후 기사: "
    f"{after_dedup:,}"
)

print(
    f"제거된 URL 중복: "
    f"{before_dedup - after_dedup:,}"
)


# ------------------------------------------------------------
# 11. 결과 구조 확인
# ------------------------------------------------------------

print()
print("article_index 컬럼:")

for i, column in enumerate(
    article_index.columns,
    start=1
):
    print(f"{i:2}. {column}")


# ------------------------------------------------------------
# 12. 샘플 확인
# ------------------------------------------------------------

print()
print("=" * 70)
print("article_index 샘플")
print("=" * 70)

display(
    article_index.head(10)
)

기존 NewsCatcher 데이터 확인
데이터 폴더:
C:\Users\황태하\Hormuz_Research\hormuz_2026_data

원본 CSV:
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\01_newscatcher_raw.csv

CSV 로드 완료
원본 레코드 수: 440,260
컬럼 수: 32

컬럼 목록:
['title', 'author', 'authors', 'journalists', 'published_date', 'published_date_precision', 'updated_date', 'updated_date_precision', 'link', 'domain_url', 'full_domain_url', 'name_source', 'is_headline', 'paid_content', 'parent_url', 'country', 'rights', 'rank', 'media', 'language', 'description', 'content', 'word_count', 'is_opinion', 'twitter_account', 'all_links', 'all_domain_links', 'id', 'score', 'duplicate_count', 'duplicate_articles_group_id', 'is_content_truncated']

article_index 생성 완료
원본 NewsCatcher 레코드: 440,260
URL 정규화 후 레코드: 440,260
URL 중복 제거 후 기사: 427,875
제거된 URL 중복: 12,385

article_index 컬럼:
 1. title_search
 2. description_search
 3. source_search
 4. published_search
 5. url
 6. language_search
 7. article_id

article_index 샘플


,title_search,description_search,source_search,published_search,url,language_search,article_id
0,US warns Iran against unsafe actions during na...,The United States has warned Iran against unsa...,wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/US_warn...,en,6e858ebeef5aa494
1,"As Iran–US tensions soar, Strait of Hormuz cau...","Dubai: The Strait of Hormuz, the narrow mouth ...",wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/As_Iran...,en,884683affb4a1f2b
2,What to know about the Strait of Hormuz as Ira...,Iran plans a military drill in the Strait of H...,sitkasentinel.com,2026-01-31 00:00:00,https://sitkasentinel.com/stories/what-to-know...,en,5e8b21085d75f44d
3,Iran's Foreign Minister Criticizes US Military...,"According to Anadolu Agency, Araghchi expresse...",turkiyenewupdates.com,2026-01-31 00:00:00,https://turkiyenewupdates.com/irans-foreign-mi...,en,6bc553fe61ec7b9e
4,Strait of Hormuz becomes centre of Iran's mili...,Iran recently warned that it will conduct a,wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/Strait_...,en,bf448b12ff11fc71
5,Explosion hits Iran's Bandar Abbas port amid t...,The port of Bandar Abbas lies on the Strait of...,wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/Explosi...,en,722ac8391f9ef3ed
6,U.S. warns Iran over unsafe naval actions,"The U.S. military has warned it will ""not tole...",anewz.tv,2026-01-31 00:00:00,https://anewz.tv/region/middle-east/17688/us-w...,en,87e3d3a0d86992da
7,"Blast in Iran port city kills 1, wounds 14 bef...","DUBAI, United Arab Emirates — An explosion tor...",wn.com,2026-01-31 00:00:00,https://article.wn.com/view/2026/01/31/Blast_i...,en,7f6698dde7b072d8
8,"Blast in Iran port city kills 1, wounds 14 bef...",An explosion has ripped through an apartment b...,sitkasentinel.com,2026-01-31 00:00:00,https://sitkasentinel.com/stories/blast-in-ira...,en,2ed76743a1b74307
9,US will 'not tolerate unsafe' actions by Iran'...,"HAMILTON, Canada\nUS Central Command (CENTCOM)...",aa.com.tr,2026-01-31 00:00:00,https://www.aa.com.tr/en/middle-east/us-will-n...,en,4c96850392df843e


## 5-1. Diffbot 이전 메타데이터 품질 필터링

현재 약 42만 건의 후보 기사 전체에 Diffbot을 호출하면 비용과 시간이 매우 커집니다.
따라서 **본문을 가져오기 전에 현재 확보된 NewsCatcher 메타데이터만으로 명백한 저품질/중복 후보를 먼저 제거**합니다.

### 이 단계에서 적용하는 기준

1. URL이 정상적인 HTTP/HTTPS 주소인지
2. 연구 기간(2026-01-01 ~ 2026-08-31) 안의 날짜인지
3. 제목이 존재하고 지나치게 짧지 않은지
4. NewsCatcher가 반환한 언어 정보가 `en`인지
5. 제목 정규화 후 완전히 동일한 제목이 반복되는지
6. 명백한 비기사/소셜/동영상 플랫폼 URL을 제외할지 여부
7. 제목/description에 호르무즈 관련 표현이 나타나는지 **관련성 점수로 기록**하되, 기본값은 강제 제외하지 않음

> 중요: **본문이 짧거나 없는지는 본문 수집 전에는 정확히 알 수 없습니다.**
> 따라서 `body_length < 100` 같은 기준은 Diffbot 수집 후 최종 검증 단계에서 적용합니다.
>
> 또한 NewsCatcher의 `lang=en`은 기사 본문 전체가 영어라는 것을 100% 보장하지 않습니다.
> 이 단계에서는 메타데이터 기반 1차 필터로 사용하고, 실제 본문 확보 후 Diffbot의 `humanLanguage`를 이용해 2차 확인합니다.

모든 필터는 `filter_*` 컬럼으로 남겨서 **어떤 기준으로 기사가 제외되었는지 재현 가능하게** 합니다.


In [2]:
%pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 11.6 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=d099b6dd600367b230ce46e54a78b3cff8de10c4c4836a2a4dbcd0539dd54321
  Stored in directory: c:\users\황태하\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# ============================================================
# 6. Pre-Diffbot 메타데이터 필터링
#    - 기존 article_index를 직접 사용
#    - NewsCatcher 재호출 없음
#    - 원본 CSV 재읽기 없음
#    - Diffbot 호출 없음
# ============================================================

import re
from urllib.parse import urlsplit


# ------------------------------------------------------------
# 0. 현재 작업 상태 확인
# ------------------------------------------------------------

print("현재 작업 폴더:", Path.cwd())

if "article_index" not in globals():
    raise RuntimeError(
        "article_index 변수가 없습니다.\n"
        "먼저 '5. NewsCatcher 결과를 연구용 기사 목록으로 정리' 셀을 실행하세요."
    )

if not isinstance(article_index, pd.DataFrame):
    raise TypeError(
        "article_index가 pandas DataFrame이 아닙니다."
    )

print(f"현재 article_index 기사 수: {len(article_index):,}")


# ------------------------------------------------------------
# 1. 필수 컬럼 확인
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "article_id",
    "url",
    "title_search",
    "description_search",
    "source_search",
    "published_search",
    "language_search",
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in article_index.columns
]

if missing_columns:
    raise KeyError(
        "article_index에 필요한 컬럼이 없습니다: "
        + ", ".join(missing_columns)
    )

print("필수 컬럼 확인 완료")


# ------------------------------------------------------------
# 2. 필터 설정
# ------------------------------------------------------------

FILTER_MIN_TITLE_CHARS = 10

EXCLUDED_DOMAINS = {
    "youtube.com",
    "youtu.be",
    "facebook.com",
    "instagram.com",
    "tiktok.com",
    "x.com",
    "twitter.com",
    "linkedin.com",
    "reddit.com",
}

HORMUZ_TERMS = (
    "hormuz",
    "strait of hormuz",
    "hormuz strait",
)


# ------------------------------------------------------------
# 3. 보조 함수
# ------------------------------------------------------------

def normalize_title_for_dedupe(text):
    """
    제목 중복 제거용 정규화.
    대소문자, URL, 특수문자, 연속 공백 차이를 제거합니다.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = re.sub(
        r"https?://\S+",
        " ",
        text
    )

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def get_domain(url):
    """
    URL에서 도메인만 추출합니다.
    """
    try:
        domain = (
            urlsplit(str(url)).netloc
            .lower()
            .split(":")[0]
        )
        return domain
    except Exception:
        return ""


def is_valid_url(url):
    """
    http / https 형식의 정상적인 URL인지 확인합니다.
    """
    if pd.isna(url):
        return False

    url = str(url).strip()

    if not url:
        return False

    try:
        parsed = urlsplit(url)

        return (
            parsed.scheme in {"http", "https"}
            and bool(parsed.netloc)
        )

    except Exception:
        return False


def is_obviously_non_latin(text):
    """
    제목/설명에 비라틴 문자가 명백하게 많이 포함되는 경우를 제외합니다.

    일부 고유명사에 비라틴 문자가 들어가는 것은 허용합니다.
    """
    if not isinstance(text, str):
        return False

    text = text.strip()

    if not text:
        return False

    # 한글
    korean = len(re.findall(r"[\uAC00-\uD7A3]", text))

    # 중국어 한자
    chinese = len(re.findall(r"[\u4E00-\u9FFF]", text))

    # 일본어 히라가나/가타카나
    japanese = len(
        re.findall(
            r"[\u3040-\u30FF]",
            text
        )
    )

    # 키릴 문자
    cyrillic = len(
        re.findall(
            r"[\u0400-\u04FF]",
            text
        )
    )

    non_latin_count = (
        korean
        + chinese
        + japanese
        + cyrillic
    )

    # 전체 문자 수
    letters = re.findall(r"[^\W\d_]", text, flags=re.UNICODE)

    if not letters:
        return False

    # 비라틴 문자가 전체 문자 중 30% 이상이면 제외
    ratio = non_latin_count / len(letters)

    return ratio >= 0.30


def relevance_score(row):
    """
    제목 + description의 호르무즈 관련성 점수.

    2 = 'Strait of Hormuz' 또는 'Hormuz Strait'
    1 = 'Hormuz' 포함
    0 = 메타데이터에서는 확인되지 않음

    중요:
    이 점수는 강제 제외 조건으로 사용하지 않습니다.
    """
    title = row.get("title_search", "")
    description = row.get("description_search", "")

    text = f"{title} {description}".lower()

    if not text.strip():
        return 0

    if (
        "strait of hormuz" in text
        or "hormuz strait" in text
    ):
        return 2

    if "hormuz" in text:
        return 1

    return 0


# ------------------------------------------------------------
# 4. 원본 article_index 보존
# ------------------------------------------------------------

article_index_all = article_index.copy()

print(
    f"URL 중복 제거 완료 상태의 기사 수: "
    f"{len(article_index_all):,}"
)


# ------------------------------------------------------------
# 5. URL 필터
# ------------------------------------------------------------

article_index_all["filter_valid_url"] = (
    article_index_all["url"]
    .apply(is_valid_url)
)


# ------------------------------------------------------------
# 6. 날짜 필터
# ------------------------------------------------------------

published_dt = pd.to_datetime(
    article_index_all["published_search"],
    errors="coerce",
    utc=True
)

start_dt = pd.Timestamp(
    START_DATE,
    tz="UTC"
)

# 2026-08-31 전체를 포함하기 위해
# 2026-09-01 미만으로 처리
end_dt_exclusive = (
    pd.Timestamp(
        END_DATE,
        tz="UTC"
    )
    + pd.Timedelta(days=1)
)

article_index_all["filter_date"] = (
    published_dt >= start_dt
) & (
    published_dt < end_dt_exclusive
)


# ------------------------------------------------------------
# 7. 제목 필터
# ------------------------------------------------------------

title_series = (
    article_index_all["title_search"]
    .fillna("")
    .astype(str)
    .str.strip()
)

article_index_all["filter_title"] = (
    title_series.str.len()
    >= FILTER_MIN_TITLE_CHARS
)


# ------------------------------------------------------------
# 8. NewsCatcher 영어 필터
# ------------------------------------------------------------

language_series = (
    article_index_all["language_search"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

article_index_all["filter_nc_language"] = (
    language_series.isin({
        "en",
        "eng",
        "english",
    })
)


# ------------------------------------------------------------
# 9. 도메인 필터
# ------------------------------------------------------------

article_index_all["domain"] = (
    article_index_all["url"]
    .apply(get_domain)
)


def domain_is_allowed(domain):
    if not domain:
        return False

    domain = domain.lower()

    # 정확히 제외 도메인
    if domain in EXCLUDED_DOMAINS:
        return False

    # 하위 도메인도 제외
    for excluded in EXCLUDED_DOMAINS:
        if domain.endswith("." + excluded):
            return False

    return True


article_index_all["filter_domain"] = (
    article_index_all["domain"]
    .apply(domain_is_allowed)
)


# ------------------------------------------------------------
# 10. 명백한 비라틴 문자 필터
# ------------------------------------------------------------

metadata_text = (
    article_index_all["title_search"]
    .fillna("")
    .astype(str)
    + " "
    + article_index_all["description_search"]
    .fillna("")
    .astype(str)
)

article_index_all["filter_obvious_nonlatin"] = (
    ~metadata_text.apply(is_obviously_non_latin)
)


# ------------------------------------------------------------
# 11. 제목 중복 제거
# ------------------------------------------------------------

article_index_all["normalized_title"] = (
    article_index_all["title_search"]
    .apply(normalize_title_for_dedupe)
)

article_index_all["filter_unique_title"] = (
    article_index_all["normalized_title"]
    .ne("")
    & ~article_index_all["normalized_title"].duplicated(
        keep="first"
    )
)


# ------------------------------------------------------------
# 12. 호르무즈 관련성 점수
# ------------------------------------------------------------

article_index_all["hormuz_relevance_score"] = (
    article_index_all
    .apply(relevance_score, axis=1)
)


# ------------------------------------------------------------
# 13. 최종 Pre-Diffbot 필터
# ------------------------------------------------------------

FILTER_COLUMNS = [
    "filter_valid_url",
    "filter_date",
    "filter_title",
    "filter_nc_language",
    "filter_domain",
    "filter_obvious_nonlatin",
    "filter_unique_title",
]

final_mask = article_index_all[FILTER_COLUMNS].all(axis=1)

article_index_filtered = (
    article_index_all[
        final_mask
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Diffbot에 전달할 최종 article_index
# ------------------------------------------------------------

article_index = article_index_filtered.copy()


# ------------------------------------------------------------
# 15. 제외 사유 기록
# ------------------------------------------------------------

rejected = (
    article_index_all[
        ~final_mask
    ]
    .copy()
    .reset_index(drop=True)
)

rejected["rejection_reason"] = ""


def add_reason(df, mask, reason):
    mask = mask.fillna(False)

    existing = (
        df.loc[mask, "rejection_reason"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df.loc[mask, "rejection_reason"] = existing.apply(
        lambda x: (
            f"{x}; {reason}"
            if x
            else reason
        )
    )


add_reason(
    rejected,
    ~rejected["filter_valid_url"],
    "invalid_url"
)

add_reason(
    rejected,
    ~rejected["filter_date"],
    "outside_date_range_or_missing_date"
)

add_reason(
    rejected,
    ~rejected["filter_title"],
    "missing_or_short_title"
)

add_reason(
    rejected,
    ~rejected["filter_nc_language"],
    "newscatcher_language_not_en"
)

add_reason(
    rejected,
    ~rejected["filter_domain"],
    "excluded_domain"
)

add_reason(
    rejected,
    ~rejected["filter_obvious_nonlatin"],
    "obvious_non_latin_metadata"
)

add_reason(
    rejected,
    ~rejected["filter_unique_title"],
    "duplicate_normalized_title"
)


# ------------------------------------------------------------
# 16. 결과 파일 저장
# ------------------------------------------------------------

FILTER_REPORT_FILE = (
    DATA_DIR
    / "03_pre_diffbot_filtered_articles.csv"
)

FILTER_REJECT_FILE = (
    DATA_DIR
    / "03_pre_diffbot_rejected_articles.csv"
)


article_index.to_csv(
    FILTER_REPORT_FILE,
    index=False,
    encoding="utf-8-sig"
)

rejected.to_csv(
    FILTER_REJECT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 17. 결과 확인
# ------------------------------------------------------------

print()
print("=" * 60)
print("PRE-DIFFBOT FILTER REPORT")
print("=" * 60)

print(
    f"필터링 전 article_index: "
    f"{len(article_index_all):,}"
)

for column, label in [
    ("filter_valid_url", "유효 URL"),
    ("filter_date", "연구 기간 내 날짜"),
    ("filter_title", "유효 제목"),
    ("filter_nc_language", "NewsCatcher 영어"),
    ("filter_domain", "제외 도메인 아님"),
    ("filter_obvious_nonlatin", "명백한 비라틴 문자 아님"),
    ("filter_unique_title", "정규화 제목 중복 아님"),
]:
    print(
        f"{label}: "
        f"{article_index_all[column].sum():,}"
    )

print(
    f"최종 Pre-Diffbot 후보: "
    f"{len(article_index):,}"
)

print(
    f"제외 기사: "
    f"{len(rejected):,}"
)

print()
print("호르무즈 관련성 점수 분포:")
print(
    article_index[
        "hormuz_relevance_score"
    ]
    .value_counts()
    .sort_index()
)

print()
print("저장 완료:")
print(FILTER_REPORT_FILE)
print(FILTER_REJECT_FILE)

print()
print("최종 article_index 컬럼:")
print(article_index.columns.tolist())

print()
print("최종 후보 미리보기:")
display(
    article_index[
        [
            "article_id",
            "title_search",
            "published_search",
            "language_search",
            "domain",
            "url",
            "hormuz_relevance_score",
        ]
    ].head(10)
)

현재 작업 폴더: d:\Users\HTH\Downloads
현재 article_index 기사 수: 427,875
필수 컬럼 확인 완료
URL 중복 제거 완료 상태의 기사 수: 427,875

PRE-DIFFBOT FILTER REPORT
필터링 전 article_index: 427,875
유효 URL: 427,875
연구 기간 내 날짜: 427,875
유효 제목: 427,400
NewsCatcher 영어: 427,875
제외 도메인 아님: 427,871
명백한 비라틴 문자 아님: 427,809
정규화 제목 중복 아님: 380,857
최종 Pre-Diffbot 후보: 380,344
제외 기사: 47,531

호르무즈 관련성 점수 분포:
hormuz_relevance_score
0    246818
1      7732
2    125794
Name: count, dtype: int64

저장 완료:
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\03_pre_diffbot_filtered_articles.csv
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\03_pre_diffbot_rejected_articles.csv

최종 article_index 컬럼:
['title_search', 'description_search', 'source_search', 'published_search', 'url', 'language_search', 'article_id', 'filter_valid_url', 'filter_date', 'filter_title', 'filter_nc_language', 'domain', 'filter_domain', 'filter_obvious_nonlatin', 'normalized_title', 'filter_unique_title', 'hormuz_relevance_score']

최종 후보 미리보기:


,article_id,title_search,published_search,language_search,domain,url,hormuz_relevance_score
0,6e858ebeef5aa494,US warns Iran against unsafe actions during na...,2026-01-31 00:00:00,en,article.wn.com,https://article.wn.com/view/2026/01/31/US_warn...,2
1,884683affb4a1f2b,"As Iran–US tensions soar, Strait of Hormuz cau...",2026-01-31 00:00:00,en,article.wn.com,https://article.wn.com/view/2026/01/31/As_Iran...,2
2,5e8b21085d75f44d,What to know about the Strait of Hormuz as Ira...,2026-01-31 00:00:00,en,sitkasentinel.com,https://sitkasentinel.com/stories/what-to-know...,2
3,6bc553fe61ec7b9e,Iran's Foreign Minister Criticizes US Military...,2026-01-31 00:00:00,en,turkiyenewupdates.com,https://turkiyenewupdates.com/irans-foreign-mi...,0
4,bf448b12ff11fc71,Strait of Hormuz becomes centre of Iran's mili...,2026-01-31 00:00:00,en,article.wn.com,https://article.wn.com/view/2026/01/31/Strait_...,2
5,722ac8391f9ef3ed,Explosion hits Iran's Bandar Abbas port amid t...,2026-01-31 00:00:00,en,article.wn.com,https://article.wn.com/view/2026/01/31/Explosi...,2
6,87e3d3a0d86992da,U.S. warns Iran over unsafe naval actions,2026-01-31 00:00:00,en,anewz.tv,https://anewz.tv/region/middle-east/17688/us-w...,2
7,7f6698dde7b072d8,"Blast in Iran port city kills 1, wounds 14 bef...",2026-01-31 00:00:00,en,article.wn.com,https://article.wn.com/view/2026/01/31/Blast_i...,2
8,4c96850392df843e,US will 'not tolerate unsafe' actions by Iran'...,2026-01-31 00:00:00,en,www.aa.com.tr,https://www.aa.com.tr/en/middle-east/us-will-n...,2
9,332fdc91251dd9bd,IRGC Denies Assassination of Navy Commander Af...,2026-01-31 00:00:00,en,www.sadanews.ps,https://www.sadanews.ps/en/news/271871.html,0


In [13]:
# ============================================================
# 7. Pre-Diffbot 후보 데이터 진단
#    - 현재 article_index 380,344건을 변경하지 않음
#    - 도메인 / 제목 중복 / 관련성 분포 분석
#    - Diffbot / Fundus / news-please 호출 없음
#    - 원본 CSV 변경 없음
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("PRE-DIFFBOT CANDIDATE DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# 1. 기본 상태 확인
# ------------------------------------------------------------

if "article_index" not in globals():
    raise RuntimeError(
        "article_index 변수가 없습니다.\n"
        "먼저 5번 셀과 6번 Pre-Diffbot 필터링 셀을 실행하세요."
    )

if not isinstance(article_index, pd.DataFrame):
    raise TypeError(
        "article_index가 pandas DataFrame이 아닙니다."
    )

diagnostic_df = article_index.copy()

print(f"현재 Pre-Diffbot 후보 기사 수: {len(diagnostic_df):,}")
print()

# ------------------------------------------------------------
# 2. 필수 컬럼 확인
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "article_id",
    "title_search",
    "description_search",
    "source_search",
    "published_search",
    "url",
    "domain",
    "hormuz_relevance_score",
    "normalized_title",
]

missing_columns = [
    col
    for col in REQUIRED_COLUMNS
    if col not in diagnostic_df.columns
]

if missing_columns:
    raise KeyError(
        "진단에 필요한 컬럼이 없습니다: "
        + ", ".join(missing_columns)
    )

print("필수 컬럼 확인 완료")
print()

# ------------------------------------------------------------
# 3. 도메인 기본 통계
# ------------------------------------------------------------

domain_counts = (
    diagnostic_df["domain"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

domain_counts = (
    domain_counts
    .replace("", "(도메인 없음)")
    .value_counts()
)

domain_summary = (
    domain_counts
    .rename_axis("domain")
    .reset_index(name="article_count")
)

domain_summary["article_ratio_pct"] = (
    domain_summary["article_count"]
    / len(diagnostic_df)
    * 100
)

domain_summary["cumulative_ratio_pct"] = (
    domain_summary["article_ratio_pct"]
    .cumsum()
)

print("=" * 70)
print("1. DOMAIN SUMMARY")
print("=" * 70)

print(
    f"고유 도메인 수: "
    f"{domain_summary['domain'].nunique():,}"
)

print(
    f"기사 수가 1건인 도메인: "
    f"{(domain_summary['article_count'] == 1).sum():,}"
)

print(
    f"기사 수가 10건 이상인 도메인: "
    f"{(domain_summary['article_count'] >= 10).sum():,}"
)

print(
    f"기사 수가 100건 이상인 도메인: "
    f"{(domain_summary['article_count'] >= 100).sum():,}"
)

print()

print("상위 30개 도메인:")
display(
    domain_summary.head(30)
)

# ------------------------------------------------------------
# 4. 도메인 집중도
# ------------------------------------------------------------

print()
print("=" * 70)
print("2. DOMAIN CONCENTRATION")
print("=" * 70)

for top_n in [10, 20, 50, 100]:
    top_count = (
        domain_summary
        .head(top_n)["article_count"]
        .sum()
    )

    top_ratio = (
        top_count
        / len(diagnostic_df)
        * 100
    )

    print(
        f"상위 {top_n:>3}개 도메인: "
        f"{top_count:>8,}건 "
        f"({top_ratio:6.2f}%)"
    )

# ------------------------------------------------------------
# 5. 제목 중복 구조 분석
# ------------------------------------------------------------

title_series = (
    diagnostic_df["normalized_title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

valid_titles = title_series[title_series != ""]

title_frequency = (
    valid_titles
    .value_counts()
)

print()
print("=" * 70)
print("3. NORMALIZED TITLE DUPLICATE ANALYSIS")
print("=" * 70)

print(
    f"유효한 정규화 제목 수: "
    f"{len(valid_titles):,}"
)

print(
    f"고유 정규화 제목 수: "
    f"{len(title_frequency):,}"
)

duplicate_title_groups = (
    title_frequency[
        title_frequency > 1
    ]
)

print(
    f"2회 이상 반복되는 제목 그룹: "
    f"{len(duplicate_title_groups):,}"
)

duplicate_article_count = (
    duplicate_title_groups.sum()
)

print(
    f"중복 제목에 포함된 기사 수: "
    f"{duplicate_article_count:,}"
)

print()

print("동일 제목 반복 횟수 분포:")
display(
    title_frequency
    .value_counts()
    .sort_index()
    .rename_axis("same_title_article_count")
    .reset_index(name="title_group_count")
    .head(20)
)

# ------------------------------------------------------------
# 6. 가장 많이 반복되는 제목 확인
# ------------------------------------------------------------

print()
print("=" * 70)
print("4. MOST REPEATED TITLES")
print("=" * 70)

top_repeated_titles = (
    title_frequency
    .head(50)
    .rename_axis("normalized_title")
    .reset_index(name="article_count")
)

display(
    top_repeated_titles
)

# ------------------------------------------------------------
# 7. 동일 제목이 여러 도메인에서 나타나는지 확인
# ------------------------------------------------------------

print()
print("=" * 70)
print("5. CROSS-DOMAIN TITLE DUPLICATION")
print("=" * 70)

title_domain_stats = (
    diagnostic_df[
        diagnostic_df["normalized_title"].fillna("").astype(str).str.strip() != ""
    ]
    .groupby("normalized_title")
    .agg(
        article_count=("article_id", "size"),
        domain_count=("domain", "nunique"),
    )
    .reset_index()
)

cross_domain_duplicate_titles = (
    title_domain_stats[
        (title_domain_stats["article_count"] > 1)
        & (title_domain_stats["domain_count"] > 1)
    ]
    .sort_values(
        ["domain_count", "article_count"],
        ascending=False
    )
)

print(
    f"여러 도메인에서 반복되는 제목 그룹: "
    f"{len(cross_domain_duplicate_titles):,}"
)

print()

print("여러 도메인에서 반복되는 제목 상위 30개:")

display(
    cross_domain_duplicate_titles.head(30)
)

# ------------------------------------------------------------
# 8. 도메인별 제목 중복률
# ------------------------------------------------------------

domain_title_stats = (
    diagnostic_df
    .groupby("domain", dropna=False)
    .agg(
        article_count=("article_id", "size"),
        unique_title_count=("normalized_title", "nunique"),
    )
    .reset_index()
)

domain_title_stats["duplicate_title_count"] = (
    domain_title_stats["article_count"]
    - domain_title_stats["unique_title_count"]
)

domain_title_stats["duplicate_title_ratio_pct"] = (
    domain_title_stats["duplicate_title_count"]
    / domain_title_stats["article_count"]
    * 100
)

domain_title_stats = (
    domain_title_stats
    .sort_values(
        "article_count",
        ascending=False
    )
)

print()
print("=" * 70)
print("6. DOMAIN-LEVEL TITLE DUPLICATION")
print("=" * 70)

print(
    "기사 수가 100건 이상인 도메인 중 "
    "제목 중복률이 높은 도메인:"
)

display(
    domain_title_stats[
        domain_title_stats["article_count"] >= 100
    ]
    .sort_values(
        "duplicate_title_ratio_pct",
        ascending=False
    )
    .head(30)
)

# ------------------------------------------------------------
# 9. 호르무즈 관련성 점수 분포
# ------------------------------------------------------------

print()
print("=" * 70)
print("7. HORMUZ RELEVANCE SCORE")
print("=" * 70)

relevance_summary = (
    diagnostic_df[
        "hormuz_relevance_score"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("hormuz_relevance_score")
    .reset_index(name="article_count")
)

relevance_summary["article_ratio_pct"] = (
    relevance_summary["article_count"]
    / len(diagnostic_df)
    * 100
)

display(
    relevance_summary
)

# ------------------------------------------------------------
# 10. 제목 / description 기본 품질 통계
# ------------------------------------------------------------

title_length = (
    diagnostic_df["title_search"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
)

description_length = (
    diagnostic_df["description_search"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
)

metadata_quality_summary = pd.DataFrame({
    "metric": [
        "title_length",
        "description_length",
    ],
    "min": [
        title_length.min(),
        description_length.min(),
    ],
    "median": [
        title_length.median(),
        description_length.median(),
    ],
    "mean": [
        title_length.mean(),
        description_length.mean(),
    ],
    "max": [
        title_length.max(),
        description_length.max(),
    ],
})

print()
print("=" * 70)
print("8. METADATA LENGTH SUMMARY")
print("=" * 70)

display(
    metadata_quality_summary
)

# ------------------------------------------------------------
# 11. description이 없는 기사
# ------------------------------------------------------------

no_description_mask = (
    diagnostic_df["description_search"]
    .isna()
    | (
        diagnostic_df["description_search"]
        .astype(str)
        .str.strip()
        == ""
    )
)

print(
    f"description이 없는 후보: "
    f"{no_description_mask.sum():,}"
)

print(
    f"description이 있는 후보: "
    f"{(~no_description_mask).sum():,}"
)

# ------------------------------------------------------------
# 12. source_search 분포
# ------------------------------------------------------------

source_counts = (
    diagnostic_df["source_search"]
    .fillna("")
    .astype(str)
    .str.strip()
)

source_counts = (
    source_counts
    .replace("", "(source 없음)")
    .value_counts()
)

source_summary = (
    source_counts
    .rename_axis("source_search")
    .reset_index(name="article_count")
)

source_summary["article_ratio_pct"] = (
    source_summary["article_count"]
    / len(diagnostic_df)
    * 100
)

print()
print("=" * 70)
print("9. SOURCE DISTRIBUTION")
print("=" * 70)

print(
    f"고유 source 수: "
    f"{source_summary['source_search'].nunique():,}"
)

display(
    source_summary.head(30)
)

# ------------------------------------------------------------
# 13. 진단 결과 저장
# ------------------------------------------------------------

DIAGNOSTIC_DOMAIN_FILE = (
    DATA_DIR
    / "04_diagnostic_domain_summary.csv"
)

DIAGNOSTIC_TITLE_FILE = (
    DATA_DIR
    / "04_diagnostic_title_summary.csv"
)

DIAGNOSTIC_DOMAIN_TITLE_FILE = (
    DATA_DIR
    / "04_diagnostic_domain_title_summary.csv"
)

DIAGNOSTIC_RELEVANCE_FILE = (
    DATA_DIR
    / "04_diagnostic_relevance_summary.csv"
)

domain_summary.to_csv(
    DIAGNOSTIC_DOMAIN_FILE,
    index=False,
    encoding="utf-8-sig"
)

top_repeated_titles.to_csv(
    DIAGNOSTIC_TITLE_FILE,
    index=False,
    encoding="utf-8-sig"
)

domain_title_stats.to_csv(
    DIAGNOSTIC_DOMAIN_TITLE_FILE,
    index=False,
    encoding="utf-8-sig"
)

relevance_summary.to_csv(
    DIAGNOSTIC_RELEVANCE_FILE,
    index=False,
    encoding="utf-8-sig"
)

print()
print("=" * 70)
print("DIAGNOSTIC FILES SAVED")
print("=" * 70)

print(DIAGNOSTIC_DOMAIN_FILE)
print(DIAGNOSTIC_TITLE_FILE)
print(DIAGNOSTIC_DOMAIN_TITLE_FILE)
print(DIAGNOSTIC_RELEVANCE_FILE)

print()
print("=" * 70)
print("진단 완료")
print("=" * 70)

print(
    f"현재 article_index 최종 후보 수: "
    f"{len(article_index):,}"
)

print(
    "※ 이 셀에서는 article_index의 행을 삭제하거나 "
    "필터링하지 않았습니다."
)

PRE-DIFFBOT CANDIDATE DIAGNOSTIC
현재 Pre-Diffbot 후보 기사 수: 380,344

필수 컬럼 확인 완료

1. DOMAIN SUMMARY
고유 도메인 수: 16,325
기사 수가 1건인 도메인: 6,276
기사 수가 10건 이상인 도메인: 3,728
기사 수가 100건 이상인 도메인: 741

상위 30개 도메인:


,domain,article_count,article_ratio_pct,cumulative_ratio_pct
0,article.wn.com,13567,3.567034,3.567034
1,news-pravda.com,8363,2.198799,5.765833
2,www.msn.com,4670,1.227836,6.993669
3,www.vietnam.vn,3475,0.913647,7.907315
4,timesofindia.indiatimes.com,2863,0.752740,8.660055
5,custommapposter.com,2845,0.748007,9.408062
6,finance.yahoo.com,2636,0.693057,10.101119
7,economictimes.indiatimes.com,2453,0.644942,10.746061
8,www.yahoo.com,2288,0.601561,11.347622
9,us.headtopics.com,2238,0.588415,11.936037



2. DOMAIN CONCENTRATION
상위  10개 도메인:   45,398건 ( 11.94%)
상위  20개 도메인:   61,435건 ( 16.15%)
상위  50개 도메인:   88,155건 ( 23.18%)
상위 100개 도메인:  118,207건 ( 31.08%)

3. NORMALIZED TITLE DUPLICATE ANALYSIS
유효한 정규화 제목 수: 380,344
고유 정규화 제목 수: 380,344
2회 이상 반복되는 제목 그룹: 0
중복 제목에 포함된 기사 수: 0

동일 제목 반복 횟수 분포:


,same_title_article_count,title_group_count
0,1,380344



4. MOST REPEATED TITLES


...


8. METADATA LENGTH SUMMARY


,metric,min,median,mean,max
0,title_length,10,70.0,75.941243,7030
1,description_length,1,303.0,295.984443,303


description이 없는 후보: 0
description이 있는 후보: 380,344

9. SOURCE DISTRIBUTION
고유 source 수: 12,623


,source_search,article_count,article_ratio_pct
0,wn.com,13567,3.567034
1,news-pravda.com,8549,2.247702
2,yahoo.com,8301,2.182498
3,substack.com,7275,1.912742
4,headtopics.com,5787,1.521517
5,indiatimes.com,5598,1.471826
6,msn.com,4670,1.227836
7,vietnam.vn,3475,0.913647
8,investing.com,3117,0.819521
9,custommapposter.com,2845,0.748007



DIAGNOSTIC FILES SAVED
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\04_diagnostic_domain_summary.csv
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\04_diagnostic_title_summary.csv
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\04_diagnostic_domain_title_summary.csv
C:\Users\황태하\Hormuz_Research\hormuz_2026_data\04_diagnostic_relevance_summary.csv

진단 완료
현재 article_index 최종 후보 수: 380,344
※ 이 셀에서는 article_index의 행을 삭제하거나 필터링하지 않았습니다.


## 6. Diffbot Article API — 본문/언론사/국가/지역 추출

앞 단계에서 **메타데이터 기준으로 선별된 `article_index`만** Diffbot 대상이 됩니다.

현재 Free 플랜에서는 월간 Extract 크레딧과 요청 속도 제한이 있으므로,
필터링 결과를 확인한 뒤 본문 수집을 시작합니다.

Diffbot Article API의 핵심 필드는 다음과 같습니다.

- `title`
- `text` = 전체 기사 본문
- `date`
- `author`
- `siteName`
- `publisherCountry`
- `publisherRegion`
- `humanLanguage`
- `resolvedPageUrl`
- `discussion`

기사 본문이 실제로 추출되었는지를 `body_length`와 `extraction_status`로 기록합니다.


In [7]:
def diffbot_article(url, include_discussion=False, timeout_ms=60000):
    """Diffbot Article API로 기사 본문을 추출합니다.

    댓글은 기본적으로 수집하지 않습니다.
    댓글이 필요할 경우 별도의 Discussion API 단계에서 보완합니다.
    """
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "discussion": "true" if include_discussion else "false",
        "paging": "true",
    }

    for attempt in range(5):
        try:
            r = requests.get(
                DIFFBOT_ARTICLE_URL,
                params=params,
                timeout=90,
            )

            if r.status_code == 429:
                if attempt == 4:
                    raise RuntimeError("Diffbot rate limit(429)입니다.")
                wait = min(60, 2 ** attempt * 2)
                print(f"  Diffbot 429 → {wait}초 후 재시도")
                time.sleep(wait)
                continue

            if 500 <= r.status_code < 600:
                if attempt == 4:
                    r.raise_for_status()
                wait = min(60, 2 ** attempt * 2)
                time.sleep(wait)
                continue

            r.raise_for_status()
            data = r.json()

            objects = data.get("objects") or []
            if not objects:
                return None, data

            return objects[0], data

        except requests.RequestException:
            if attempt == 4:
                raise
            wait = min(60, 2 ** attempt * 2)
            time.sleep(wait)

    return None, None


def flatten_article(article_id, source_url, obj):
    if not obj:
        return {
            "article_id": article_id,
            "source_url": source_url,
            "extraction_status": "no_object",
            "body": "",
            "body_length": 0,
        }

    body = obj.get("text") or ""

    return {
        "article_id": article_id,
        "source_url": source_url,
        "resolved_url": obj.get("resolvedPageUrl"),
        "title": obj.get("title"),
        "body": body,
        "body_length": len(body),
        "published_at": obj.get("date"),
        "estimated_date": obj.get("estimatedDate"),
        "author": obj.get("author"),
        "source": obj.get("siteName"),
        "publisher_country": obj.get("publisherCountry"),
        "publisher_region": obj.get("publisherRegion"),
        "language": obj.get("humanLanguage"),
        "num_pages": obj.get("numPages"),
        "extraction_status": (
            "success" if len(body.strip()) >= 100 else "short_or_empty"
        ),
    }


## 7. 본문 수집 실행

중단되더라도 이미 저장된 결과를 다시 요청하지 않도록
`articles_partial.csv`에 주기적으로 저장합니다.

처음에는 **API 테스트 비용을 아끼기 위해 `TEST_LIMIT=20`**으로 실행합니다.
20건의 결과가 정상임을 확인한 뒤 `TEST_LIMIT=None`으로 변경하여 전체 수집합니다.

### 댓글

댓글은 프로젝트의 필수 데이터가 아니므로 **기본적으로 수집하지 않습니다.**
기사 본문 수집이 우선입니다.


In [8]:
# Free 플랜 안전 설정
# 테스트 후 전체 수집하려면 TEST_LIMIT=None으로 변경합니다.
TEST_LIMIT = 20

# 너무 빠른 호출을 방지하기 위한 기본 대기 시간.
# Free 플랜의 rate limit에 맞추어 충분한 간격을 둡니다.
DIFFBOT_REQUEST_INTERVAL = 13

partial_file = DATA_DIR / "articles_partial.csv"
failed_file = DATA_DIR / "failed_urls_partial.csv"

if partial_file.exists():
    done_df = pd.read_csv(partial_file)
    done_ids = set(done_df["article_id"].astype(str))
else:
    done_df = pd.DataFrame()
    done_ids = set()

targets = article_index[
    ~article_index["article_id"].astype(str).isin(done_ids)
].copy()

if TEST_LIMIT is not None:
    targets = targets.head(TEST_LIMIT)

print("Diffbot 본문 수집 대상:", len(targets))

article_rows = []
failed_rows = []

for _, row in tqdm(
    targets.iterrows(),
    total=len(targets),
    desc="Diffbot Article",
):
    try:
        obj, raw = diffbot_article(
            row["url"],
            include_discussion=False,
        )

        article_rows.append(
            flatten_article(row["article_id"], row["url"], obj)
        )

    except Exception as e:
        failed_rows.append({
            "article_id": row["article_id"],
            "source_url": row["url"],
            "error": str(e),
        })

    # Free 플랜 rate limit을 고려해 호출 사이에 충분한 간격을 둡니다.
    time.sleep(DIFFBOT_REQUEST_INTERVAL)

new_articles_df = pd.DataFrame(article_rows)

if len(done_df):
    articles_df = pd.concat(
        [done_df, new_articles_df],
        ignore_index=True,
    )
else:
    articles_df = new_articles_df.copy()

if len(articles_df):
    articles_df = articles_df.drop_duplicates(
        "article_id"
    ).reset_index(drop=True)

articles_df.to_csv(
    partial_file,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(failed_rows).to_csv(
    failed_file,
    index=False,
    encoding="utf-8-sig",
)

print("현재 Article 결과:", len(articles_df))

if len(articles_df):
    print(
        "본문 100자 이상:",
        (articles_df["body_length"] >= 100).sum(),
    )
    print(
        "Diffbot humanLanguage=en:",
        (
            articles_df["language"]
            .fillna("")
            .astype(str)
            .str.lower()
            .eq("en")
        ).sum(),
    )

print("이번 실행 실패:", len(failed_rows))


Diffbot Article:   5%|▌         | 1/20 [00:05<01:37,  5.11s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  10%|█         | 2/20 [00:30<05:08, 17.14s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  15%|█▌        | 3/20 [00:52<05:28, 19.34s/it]

...

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도
  Diffbot 429 → 8초 후 재시도


Diffbot Article:  90%|█████████ | 18/20 [07:03<00:44, 22.33s/it]

  Diffbot 429 → 2초 후 재시도
  Diffbot 429 → 4초 후 재시도


Diffbot Article: 100%|██████████| 20/20 [07:23<00:00, 22.19s/it]

현재 Article 결과: 20
본문 100자 이상: 17
이번 실행 실패: 0


## 7-1. Diffbot 본문 품질 필터

본문을 실제로 받은 뒤에만 확인할 수 있는 항목을 검사합니다.

- 본문이 비어 있는지
- 본문 길이가 100자 미만인지
- Diffbot의 `humanLanguage`가 영어인지
- 제목/본문이 실제로 존재하는지

`100자`는 프로젝트의 최종 텍스트마이닝 후보를 위한 **보수적인 최소 기준**입니다.
필요하면 200자, 300자 등으로 변경할 수 있습니다.

이 단계에서는 원본 `articles_df`를 삭제하지 않고,
`articles_quality_filtered`를 별도로 만들어 원본 결과를 보존합니다.


In [ ]:
MIN_BODY_LENGTH = 100

articles_quality_filtered = articles_df.copy()

articles_quality_filtered["quality_has_body"] = (
    articles_quality_filtered["body"].fillna("").astype(str).str.strip().ne("")
)

articles_quality_filtered["quality_body_length"] = (
    articles_quality_filtered["body"].fillna("").astype(str).str.len()
)

articles_quality_filtered["quality_body_long_enough"] = (
    articles_quality_filtered["quality_body_length"] >= MIN_BODY_LENGTH
)

articles_quality_filtered["quality_language_en"] = (
    articles_quality_filtered["language"]
    .fillna("")
    .astype(str)
    .str.lower()
    .eq("en")
)

articles_quality_filtered["quality_title_exists"] = (
    articles_quality_filtered["title"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

final_analysis_df = articles_quality_filtered[
    articles_quality_filtered["quality_has_body"] &
    articles_quality_filtered["quality_body_long_enough"] &
    articles_quality_filtered["quality_language_en"] &
    articles_quality_filtered["quality_title_exists"]
].copy().reset_index(drop=True)

QUALITY_FILTER_FILE = DATA_DIR / "04_articles_quality_filtered.csv"
articles_quality_filtered.to_csv(
    QUALITY_FILTER_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("===== POST-DIFFBOT BODY QUALITY REPORT =====")
print("Diffbot 결과:", len(articles_quality_filtered))
print("본문 존재:", articles_quality_filtered["quality_has_body"].sum())
print(f"본문 {MIN_BODY_LENGTH}자 이상:", articles_quality_filtered["quality_body_long_enough"].sum())
print("Diffbot humanLanguage=en:", articles_quality_filtered["quality_language_en"].sum())
print("제목 존재:", articles_quality_filtered["quality_title_exists"].sum())
print("최종 텍스트마이닝 후보:", len(final_analysis_df))
print(f"\n저장: {QUALITY_FILTER_FILE}")


## 8. 댓글은 선택 사항

댓글은 본 연구의 필수 수집 대상이 아닙니다.

기본 실행에서는 Article API에 `discussion=false`를 사용하여
기사 본문 수집에 집중합니다.

댓글 분석이 필요해졌을 때만 아래의 Discussion API를 별도로 실행할 수 있습니다.


In [ ]:
def extract_comments_from_article_object(article_id, obj):
    rows = []

    if not obj:
        return rows

    discussion = obj.get("discussion")
    if not discussion:
        return rows

    posts = discussion.get("posts") or []

    for post in posts:
        text = post.get("text") or ""

        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": text,
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 9. 댓글을 확실하게 별도 수집하고 싶을 때

Article API의 `discussion`이 비어 있는 사이트는 Discussion API를 한 번 더 호출할 수 있습니다.
다만 이것은 **모든 기사에 무조건 두 번 요청하지 않도록** 설계하는 것이 좋습니다.

아래 함수는 필요할 때만 Discussion API를 호출합니다.


In [ ]:
def diffbot_discussion(url, timeout_ms=60000):
    params = {
        "token": DIFFBOT_API_KEY,
        "url": url,
        "timeout": timeout_ms,
        "maxPages": "all",
    }

    r = requests.get(DIFFBOT_DISCUSSION_URL, params=params, timeout=90)

    if r.status_code == 429:
        raise RuntimeError("Diffbot Discussion rate limit(429)입니다.")

    r.raise_for_status()
    data = r.json()

    objects = data.get("objects") or []
    return objects[0] if objects else None

def flatten_discussion(article_id, discussion_obj):
    rows = []

    if not discussion_obj:
        return rows

    for post in discussion_obj.get("posts") or []:
        rows.append({
            "article_id": article_id,
            "comment_id": post.get("id"),
            "parent_comment_id": post.get("parentId"),
            "comment_text": post.get("text"),
            "comment_author": post.get("author"),
            "comment_date": post.get("date"),
            "comment_language": post.get("humanLanguage"),
            "comment_url": post.get("pageUrl"),
        })

    return rows


## 10. 현재 Article 결과에서 댓글이 없는 기사만 Discussion API로 보완

이 단계는 API 비용이 추가될 수 있으므로 필요할 때 실행합니다.


In [ ]:
RUN_DISCUSSION_BACKFILL = False  # 필요할 때만 True

comments_rows = []

if RUN_DISCUSSION_BACKFILL:
    for _, row in tqdm(
        articles_df[articles_df["source_url"].notna()].iterrows(),
        total=articles_df["source_url"].notna().sum(),
        desc="Diffbot Discussion",
    ):
        try:
            discussion = diffbot_discussion(row["source_url"])
            comments_rows.extend(
                flatten_discussion(row["article_id"], discussion)
            )
        except Exception as e:
            print("댓글 실패:", row["source_url"], e)

        time.sleep(0.2)

comments_df = pd.DataFrame(comments_rows)

if len(comments_df):
    comments_df = comments_df.drop_duplicates(
        subset=["article_id", "comment_id", "comment_text"]
    )
    comments_df.to_csv(
        COMMENTS_FILE,
        index=False,
        encoding="utf-8-sig",
    )
    print("댓글 수:", len(comments_df))
else:
    print(
        "댓글 수집을 실행하지 않았습니다. "
        "필요한 경우 RUN_DISCUSSION_BACKFILL=True로 변경하세요."
    )


## 11. 수집 결과 검증

최종 수집 결과에서 다음 항목을 확인합니다.

- raw 검색 결과 수
- URL 중복 제거 후 기사 수
- Diffbot 본문 추출 성공 수
- 본문 100자 미만 수
- 영어 기사 수
- 언론사 국가 미상 수
- 언론사명 미상 수
- 수집된 날짜 범위

특히 **2026-01-01부터 2026-08-31까지 실제 날짜가 분포하는지** 확인하는 것이 중요합니다.


In [9]:
# 전체 저장 articles_df
articles_df.to_csv(
    ARTICLES_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("===== ARTICLE COLLECTION REPORT =====")
print("raw 기사 수:", len(raw_df))
print("URL 중복 제거 후 기사 수:", len(article_index_all))
print("사전 필터링 후 Diffbot 후보:", len(article_index))
print("Diffbot 기사 수:", len(articles_df))

if len(articles_df):
    print(
        "본문 100자 이상:",
        (articles_df["body_length"] >= 100).sum(),
    )
    print(
        "Diffbot humanLanguage=en:",
        (
            articles_df["language"]
            .fillna("")
            .astype(str)
            .str.lower()
            .eq("en")
        ).sum(),
    )

print("최종 텍스트마이닝 후보:", len(final_analysis_df) if "final_analysis_df" in globals() else 0)

if len(articles_df):
    dates = pd.to_datetime(
        articles_df["published_at"],
        errors="coerce",
    ).dropna()

    if len(dates):
        print("Diffbot 날짜 범위:", dates.min(), "~", dates.max())


===== ARTICLE COLLECTION REPORT =====
raw 기사 수: 440260
URL 중복 제거 후 기사 수: 427874
Diffbot 기사 수: 20
본문 100자 이상: 17
본문 100자 미만: 3
언어 en: 17
국가 미상: 16
언론사 미상: 3
Diffbot 날짜 최소: 2026-01-30 00:00:00
Diffbot 날짜 최대: 2026-09-11 00:00:00


,article_id,title,source,publisher_country,publisher_region,language,published_at,body_length,extraction_status
0,6e858ebeef5aa494,US warns Iran against unsafe actions during na...,wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",471,success
1,884683affb4a1f2b,"As Iran–US tensions soar, Strait of Hormuz cau...",wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",486,success
2,5e8b21085d75f44d,What to know about the Strait of Hormuz as Ira...,Daily Sitka Sentinel,NaN,NaN,en,"Wed, 18 Feb 2026 03:35:10 GMT",4451,success
3,6bc553fe61ec7b9e,Iran’s Foreign Minister Criticizes US Military...,Türkiye New Updates,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",2225,success
4,bf448b12ff11fc71,NaN,NaN,NaN,NaN,NaN,NaN,0,no_object
5,722ac8391f9ef3ed,Explosion hits Iran's Bandar Abbas port amid t...,wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",195,success
6,87e3d3a0d86992da,U.S. warns Iran over unsafe naval actions,anewz.tv,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",2477,success
7,7f6698dde7b072d8,"Blast in Iran port city kills 1, wounds 14 bef...",wn.com,NaN,NaN,en,"Sat, 31 Jan 2026 00:00:00 GMT",395,success
8,2ed76743a1b74307,NaN,NaN,NaN,NaN,NaN,NaN,0,no_object
9,4c96850392df843e,US will 'not tolerate unsafe' actions by Iran'...,Anadolu Ajansi,Turkey,Western Asia,en,"Fri, 30 Jan 2026 22:52:07 GMT",2245,success


## 12. Excel 파일 생성

분석용으로는 CSV가 더 안정적이고,
확인/제출용으로는 Excel을 같이 만들어 둡니다.


In [10]:
excel_path = DATA_DIR / "hormuz_2026_articles_comments.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    articles_df.to_excel(writer, sheet_name="articles", index=False)

    if "final_analysis_df" in globals():
        final_analysis_df.to_excel(
            writer,
            sheet_name="final_textmining",
            index=False,
        )

    if "article_index_all" in globals():
        article_index_all.head(100000).to_excel(
            writer,
            sheet_name="pre_filter_sample",
            index=False,
        )

    if COMMENTS_FILE.exists():
        comments_for_excel = pd.read_csv(COMMENTS_FILE)
        comments_for_excel.to_excel(writer, sheet_name="comments", index=False)

print("저장 완료:", excel_path)


저장 완료: hormuz_2026_data\hormuz_2026_articles_comments.xlsx


# 13. 기존 텍스트마이닝 코드와 연결

여기부터는 기존 노트북의 NLTK 전처리 코드를 연결하면 됩니다.

중요한 점은 `body`를 분석 대상으로 사용하고,
`title`은 별도 분석 컬럼으로 보존하는 것입니다.

또한 `hormuz`, `strait`은 연구 질문에 따라 불용어로 제거할 수 있지만,
`iran`, `iranian`, `israel`, `israeli`, `us`, `america` 등을 무조건 제거하면
국가/행위자 프레임 분석에 필요한 정보가 사라질 수 있으므로 신중하게 결정합니다.


In [ ]:
# 텍스트마이닝 시작용 최소 코드
# 본문 품질 필터를 통과한 기사만 분석 대상으로 사용합니다.
analysis_df = final_analysis_df.copy()

analysis_df["text_for_mining"] = (
    analysis_df["title"].fillna("") + " " +
    analysis_df["body"].fillna("")
)

print("텍스트마이닝 대상 기사:", len(analysis_df))

analysis_df[[
    "article_id",
    "title",
    "source",
    "language",
    "body_length",
]].head()


## 14. 중단 후 재시작

`articles_partial.csv`가 남아 있으면 이미 처리된 `article_id`는 다시 요청하지 않습니다.

### 권장 실행 순서

1. NewsCatcher 전체 기간 수집
2. URL 중복 제거 결과 확인
3. **5-1. Diffbot 이전 메타데이터 필터 실행**
4. `PRE-DIFFBOT FILTER REPORT`에서 최종 Diffbot 후보 수 확인
5. `TEST_LIMIT = 20`으로 Diffbot 본문 추출 테스트
6. 20건 결과와 `body_length`, `humanLanguage` 확인
7. 문제가 없으면 `TEST_LIMIT = None`으로 변경
8. `RUN_DISCUSSION_BACKFILL = False` 유지
9. 본문 수집 후 `7-1. Diffbot 본문 품질 필터` 실행
10. `final_analysis_df`를 기존 텍스트마이닝 분석에 사용

> `articles_partial.csv`는 실제 Diffbot 요청 결과가 저장되는 체크포인트입니다.
> `03_pre_diffbot_filtered_articles.csv`는 Diffbot 요청 전에 선정된 후보 목록입니다.
> `03_pre_diffbot_rejected_articles.csv`는 제외된 기사와 제외 사유를 기록합니다.


In [11]:
import os
import requests

DIFFBOT_API_KEY = os.getenv("DIFFBOT_API_KEY")

url = "https://api.diffbot.com/v4/account"

response = requests.get(
    url,
    params={
        "token": DIFFBOT_API_KEY,
        "days": 31
},
    timeout=30
)

print("HTTP 상태 코드:", response.status_code)

if response.ok:
    data = response.json()

    print("플랜:", data.get("plan"))
    print("상태:", data.get("status"))
    print("월간 포함 크레딧:", data.get("planCredits"))

    usage = data.get("usage", [])

    if usage:
        total_credits = sum(
            item.get("credits", 0)
            for item in usage
        )

        total_extractions = sum(
            item.get("extractions", 0)
            for item in usage
        )

        print("최근 31일 사용 크레딧:", total_credits)
        print("최근 31일 Extract 호출:", total_extractions)

        print("\n최근 사용량:")
        for item in usage[:10]:
            print(
                item.get("date"),
                "credits =", item.get("credits", 0),
                "extractions =", item.get("extractions", 0)
            )
else:
    print("Diffbot API 오류")
    print(response.text)

HTTP 상태 코드: 200
플랜: kgfree
상태: active
월간 포함 크레딧: 10000
최근 31일 사용 크레딧: 20
최근 31일 Extract 호출: 20

최근 사용량:
2026-09-11 credits = 20 extractions = 20
2026-09-10 credits = 0 extractions = 0
2026-09-09 credits = 0 extractions = 0
2026-09-08 credits = 0 extractions = 0
2026-09-07 credits = 0 extractions = 0
2026-09-06 credits = 0 extractions = 0
2026-09-05 credits = 0 extractions = 0
2026-09-04 credits = 0 extractions = 0
2026-09-03 credits = 0 extractions = 0
2026-09-02 credits = 0 extractions = 0
